*This notebook supports training of the **QF Map Generator** and **Predictor** using Google Cloud TPU.*

## Setup

In [ ]:
!git clone https://github.com/299-792458/qf-map-generator
!mv qf-map-generator/* .

In [ ]:
import os
import tensorflow as tf
import tensorflow_datasets as tfds
from network import *
from network.model import JPEGEndToEnd

In [ ]:
resolver = tf.distribute.cluster_resolver.TPUClusterResolver()
tf.config.experimental_connect_to_cluster(resolver)
tf.tpu.experimental.initialize_tpu_system(resolver)
strategy = tf.distribute.TPUStrategy(resolver)

In [ ]:
#@title GCS Authentication

from google.colab import auth
auth.authenticate_user()

In [ ]:
#@title Enter Path to

#@markdown > *Dataset directory*
DATA_DIR = ''  #@param {type: 'string'}
#@markdown > *the root of Log directory*
LOGDIR_ROOT = '.'  #@param {type: 'string'}
#@markdown > *the saved bpp estimator model*
BPP_ESTIMATOR_PATH = ''  #@param {type: 'string'}

In [ ]:
#@title Hyperparameters

INPUT_SIZE = 224  #@param {type: 'number'}
NUM_STEPS = 1000  #@param {type: 'number'}
EPOCHS = 50  #@param {type: 'number'}
BATCH_SIZE = 64  #@param {type: 'number'}
LEARNING_RATE = 1e-4  #@param {type: 'number'}
RATE_LOSS_COEF = 4  #@param {type: 'number'}
QF_BIAS = 0  #@param {type: 'number'}
CLIP_QF_MIN = 2  #@param {type: 'number'}
CLIP_QF_MAX = 50  #@param {type: 'number'}

#@title Check if you want to save the model checkpoint after every epoch
save_ckpt = False #@param {type:'boolean'}

## Train

In [ ]:
#@title Load Dataset

from network.preprocess import preprocess_train, preprocess_valid
import functools


train_dataset = tfds.load('imagenet2012',
                          split='train',
                          decoders={'image': tfds.decode.SkipDecoding()},
                          as_supervised=True,
                          data_dir=DATA_DIR)

valid_dataset = tfds.load('imagenet2012',
                          split='validation',
                          decoders={'image': tfds.decode.SkipDecoding()},
                          as_supervised=True,
                          data_dir=DATA_DIR)

func_train = functools.partial(preprocess_train, size=INPUT_SIZE)
func_valid = functools.partial(preprocess_valid, size=INPUT_SIZE)

train_data = (train_dataset.map(func_train, -1)
                              .shuffle(5000)
                              .repeat(-1)
                              .batch(BATCH_SIZE, drop_remainder=True)
                              .prefetch(-1))

valid_data = (valid_dataset.map(func_valid, -1)
                              .batch(1000)
                              .prefetch(-1))

In [ ]:
#@title Build and Compile Model

with strategy.scope():
    bpp_estimator = tf.keras.models.load_model(BPP_ESTIMATOR_PATH)
    classifier = tf.keras.applications.ResNet50(weights='imagenet')
    classifier.compile(metrics=['accuracy'])

classifier.preprocess = tf.keras.applications.resnet.preprocess_input

for layer in bpp_estimator.layers:
    layer.trainable = False
for layer in classifier.layers:
    layer.trainable = False

with strategy.scope():
    model = JPEGEndToEnd(bpp_estimator=bpp_estimator,
                         classifier=classifier,
                         qf_bias=QF_BIAS,
                         clip_qf_min=CLIP_QF_MIN,
                         clip_qf_max=CLIP_QF_MAX,
                         rate_loss_coef=RATE_LOSS_COEF,
                         )
    model.compile(
        optimizer=tf.keras.optimizers.Adam(LEARNING_RATE),
        loss=tf.keras.losses.SparseCategoricalCrossentropy(),
        metrics=['accuracy']
        )

In [ ]:
#@title Callbacks

ckpt_dir = os.path.join(LOGDIR_ROOT, 'ckpt/')
checkpoint_options = tf.train.CheckpointOptions(experimental_io_device='/job:localhost')

ckpt_cb = tf.keras.callbacks.ModelCheckpoint(
    filepath = ckpt_dir + '{epoch:04d}.ckpt',
    save_weights_only=True,
    save_best_only=False,
    options=checkpoint_options,
    verbose=1,
    save_freq='epoch')


callbacks = []

if save_ckpt:
    callbacks.append(ckpt_cb)
    print(f'Checkpoints will be saved in \'{ckpt_dir}\'\n')

In [ ]:
history = model.fit(train_data,
                    steps_per_epoch=NUM_STEPS,
                    epochs=EPOCHS,
                    callbacks=callbacks,
                    verbose=1
                    )